# 4. Model

This notebook assesses the model, tunes it, trains its final version and writes the Kaggle submission. The model is a logistic regression on the columns kept by the feature-selection strategy saved in `logs/feature_selection.json` (the multicollinearity filter at 0.90, then an L1-penalised logistic regression with C=0.1).

- **Assessment:** the same 7 temporal splits as the pre-processing and feature-selection notebooks. Inside each split, the recipe, the feature selection, the model and the cut-off are fitted on the training rows only; the validation rows are only scored.
- **Tuning:** the settings of the logistic regression are searched on the same splits, as in the course practical (week 6), and the search is then scored on splits that took no part in the choice.
- **Final model:** the same steps fitted once on every training row, then used to predict every row of `test.csv`.

## 4.1 Setup

In [1]:
# Standard library
import json
import sys
import warnings
from pathlib import Path

# Arrays, tables, the model and its metrics
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

# The group's own functions (src/3_Feature_selection/preprocessing.py and selection.py)
sys.path.append("../../src/3_Feature_selection")
from preprocessing import (RANDOM_STATE, fold_matrices, load_data, paired_delta, purchase_dates,
                           target, temporal_splits)
from selection import make_strategy_selector

# Functions of the model notebooks (src/4_Model/model.py)
sys.path.append("../../src/4_Model")
from model import (CUTOFFS, baseline_rule, best_cutoff, f1_by_cutoff, fit_with_cutoff,
                   score_configurations, walk_forward)

warnings.filterwarnings("ignore", category=UserWarning)  # unknown levels in validation, as in feature selection

In [2]:
# The feature-selection strategy: the candidate columns and the selector's settings
strategy = json.loads(Path("../../logs/feature_selection.json").read_text(encoding="utf-8"))
numeric = strategy["candidate_columns"]["numeric"]
categorical = strategy["candidate_columns"]["categorical"]
selector = strategy["selector"]

train_clientes = load_data()
# The splits need the raw date parts (year, month, day), so they are built before the calendar
# columns are replaced by their sin/cos components, as in the feature-selection notebook
time_splits = temporal_splits(train_clientes)
for col, period in [("purchase_month", 12.0), ("purchase_day", 31.0), ("purchase_dayofweek", 7.0)]:
    angle = 2.0 * np.pi * train_clientes[col].astype(float) / period
    train_clientes[f"{col}_sin"] = np.sin(angle)
    train_clientes[f"{col}_cos"] = np.cos(angle)
frame = train_clientes[numeric + categorical + [target]]

# The pre-processing team's rules (1.6, 1.7.3, 1.7.4 and 1.8), as in the feature-selection notebook
labels_as_numbers = ["is_first_purchase", "is_weekend", "since_2015_or_earlier", "purchase_year",
                     "customer_since_year", "purchase_month_sin", "purchase_month_cos",
                     "purchase_day_sin", "purchase_day_cos", "purchase_dayofweek_sin",
                     "purchase_dayofweek_cos"]
recipe = {"numeric": numeric, "categorical": categorical,
          "fill": [(numeric, SimpleImputer(strategy="median")),
                   (categorical, SimpleImputer(strategy="most_frequent"))],
          "log1p": [c for c in numeric
                    if c not in labels_as_numbers and not (train_clientes[c] < 0).any()],
          "rare": {"transport_method": 50, "payment_type": 20, "city": 30, "municipality": 30,
                   "postal_area": 20}}
print(f"{len(frame):,} training rows | {len(numeric)} numeric and {len(categorical)} categorical"
      f" candidate columns | {len(time_splits)} temporal splits")

6,469 training rows | 39 numeric and 8 categorical candidate columns | 7 temporal splits


## 4.2 Assessment on the Temporal Splits

Each split trains on the past and validates on the months after it, with the 90-day gap of section 1.5.2. Inside each split, everything that learns from the data is fitted on the training rows:

1. **The recipe** (fill, `log1p`, rare levels, one-hot encoding and scaling) with `fold_matrices`.
2. **The feature selection:** the multicollinearity filter (0.90), then L1, with the settings saved in the strategy.
3. **The logistic regression** on the columns the selection keeps.
4. **The cut-off:** the value with the highest F1 on the training rows. F1 is the competition metric, and when most customers come back a cut-off below 0.5 catches more of them; choosing it on the training rows keeps the validation rows out of that choice.

The validation rows are scored with F1 at that cut-off, AP and ROC-AUC (how well the probabilities rank the customers, without a cut-off), balanced accuracy, precision and recall. The training F1 sits beside the validation F1, so a wide gap would show overfitting.

In [3]:
def make_selector():
    """A fresh, unfitted selector with the strategy saved by the feature-selection notebook:
    the multicollinearity filter (0.90), then L1."""
    return make_strategy_selector(selector)


def make_model():
    """A fresh, unfitted model: the logistic regression used throughout the project."""
    return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

In [4]:
rows, curves, pooled, prepared = [], [], [], []
dates = purchase_dates(train_clientes)
for train_rows, validation_rows in time_splits:
    train, validation = frame.iloc[train_rows], frame.iloc[validation_rows]
    A_train, A_validation, _ = fold_matrices(train, validation, recipe)
    keep = make_selector().fit(A_train, train[target]).get_support()
    # The prepared matrices are kept for the tuning of 4.5
    prepared.append((A_train[:, keep], A_validation[:, keep], train[target], validation[target]))
    # The cut-off is chosen on the training rows only
    model, p_train, cutoff = fit_with_cutoff(make_model(), A_train[:, keep], train[target])
    p_validation = model.predict_proba(A_validation[:, keep])[:, 1]

    y, predicted = validation[target], (p_validation >= cutoff).astype(int)
    rows.append({"validation from": dates.iloc[validation_rows].min().date(),
                 "until": dates.iloc[validation_rows].max().date(),
                 "columns": int(keep.sum()), "cut-off": cutoff,
                 "train F1": f1_score(train[target], p_train >= cutoff),
                 "F1": f1_score(y, predicted),
                 "AP": average_precision_score(y, p_validation),
                 "ROC-AUC": roc_auc_score(y, p_validation),
                 "balanced accuracy": balanced_accuracy_score(y, predicted),
                 "precision": precision_score(y, predicted),
                 "recall": recall_score(y, predicted),
                 # The same model at the default cut-off, and the two floors of section 4.3
                 "repeat rate": y.mean(),
                 "F1 at 0.5": f1_score(y, p_validation >= 0.5),
                 "always 1": f1_score(y, np.ones(len(y), dtype=int)),
                 "baseline rule": f1_score(y, baseline_rule(validation))})
    curves.append(pd.DataFrame({"cut-off": CUTOFFS,
                                "train F1": f1_by_cutoff(train[target], p_train),
                                "validation F1": f1_by_cutoff(y, p_validation)}))
    pooled.append(pd.DataFrame({"actual": y.to_numpy(), "predicted": predicted}))

assessment = pd.DataFrame(rows)
measures = ["train F1", "F1", "AP", "ROC-AUC", "balanced accuracy", "precision", "recall"]
print(assessment[["validation from", "until", "columns", "cut-off"] + measures]
      .round(3).to_string(index=False))
print("\nmean:", assessment[measures].mean().round(4).to_dict())
print("SEM: ", (assessment[measures].std(ddof=1) / np.sqrt(len(assessment))).round(4).to_dict())

validation from      until  columns  cut-off  train F1    F1    AP  ROC-AUC  balanced accuracy  precision  recall
     2019-06-09 2020-03-28       23     0.42     0.769 0.741 0.837    0.761              0.682      0.708   0.776
     2020-03-28 2021-03-06       32     0.43     0.763 0.730 0.823    0.724              0.652      0.722   0.738
     2021-03-06 2021-11-13       27     0.44     0.766 0.769 0.854    0.815              0.741      0.750   0.788
     2021-11-13 2022-10-02       33     0.42     0.762 0.721 0.825    0.796              0.682      0.644   0.820
     2022-10-04 2023-07-28       37     0.41     0.763 0.728 0.819    0.786              0.699      0.666   0.803
     2023-07-29 2024-04-13       37     0.40     0.759 0.791 0.874    0.803              0.708      0.771   0.811
     2024-04-13 2025-02-15       37     0.40     0.762 0.801 0.863    0.806              0.696      0.735   0.879

mean: {'train F1': 0.7636, 'F1': 0.7543, 'AP': 0.8421, 'ROC-AUC': 0.7845, 'balanced acc

**Conclusions:**
- Over the 7 splits the model reaches **F1 0.754 ± 0.012** (mean ± SEM) at its cut-off, with AP 0.842, ROC-AUC 0.785 and balanced accuracy 0.694. The training F1 (0.764) stays close to the validation F1, so the model is not overfitting.
- F1 depends on the period: from 0.721 in the block starting in November 2021 to 0.801 in the latest block (April 2024 to February 2025), the one closest to the test period.
- The cut-off chosen on the training rows is stable, between 0.40 and 0.44, and the feature selection keeps between 23 and 37 columns as the training rows grow.

## 4.3 Against the Floors

F1 ignores the customers correctly left out, so a high F1 means little without a floor beside it. The model is compared, split by split, with:

- **always 1:** predicting a repeat purchase for every customer, the floor of the course (`DummyClassifier` with the majority class);
- **the baseline rule:** the baseline in `sample_submission.csv`, which predicts a repeat purchase when `purchase_count_365d >= 3` (the first line below checks it on the test rows); on the public leaderboard it scores 0.7794;
- **the same model at the default cut-off of 0.5.**

In [5]:
# The competition's baseline is a rule: every row of sample_submission.csv follows it
sample = pd.read_csv("../../data/sample_submission.csv")
test_raw = pd.read_csv("../../data/test.csv").merge(sample, on="ID")
follows = (baseline_rule(test_raw) == test_raw[target]).mean()
print(f"sample_submission rows that follow purchase_count_365d >= 3: {follows:.0%}\n")

compared = ["always 1", "baseline rule", "F1 at 0.5", "F1"]
print(assessment[["validation from", "repeat rate"] + compared].round(3).to_string(index=False))
print("\nmean:", assessment[compared].mean().round(4).to_dict())
for name in ("baseline rule", "F1 at 0.5"):
    r = paired_delta(assessment["F1"], assessment[name])
    print(f"model at its cut-off minus {name}: dF1 {r['delta']:+.4f} +/- {r['SEM']:.4f},"
          f" better on {r['wins']} -> {r['verdict']}")

sample_submission rows that follow purchase_count_365d >= 3: 100%

validation from  repeat rate  always 1  baseline rule  F1 at 0.5    F1
     2019-06-09        0.563     0.720          0.733      0.729 0.741
     2020-03-28        0.604     0.753          0.745      0.710 0.730
     2021-03-06        0.537     0.699          0.740      0.748 0.769
     2021-11-13        0.502     0.668          0.719      0.731 0.721
     2022-10-04        0.502     0.668          0.728      0.721 0.728
     2023-07-29        0.622     0.767          0.771      0.778 0.791
     2024-04-13        0.605     0.754          0.787      0.780 0.801

mean: {'always 1': 0.7186, 'baseline rule': 0.7461, 'F1 at 0.5': 0.7424, 'F1': 0.7543}
model at its cut-off minus baseline rule: dF1 +0.0081 +/- 0.0054, better on 6/7 -> inside the noise
model at its cut-off minus F1 at 0.5: dF1 +0.0119 +/- 0.0041, better on 6/7 -> better


**Conclusions:**
- The competition's baseline is exactly the rule `purchase_count_365d >= 3`: every row of `sample_submission.csv` follows it.
- At its cut-off, the model scores above both floors, always 1 (0.719) and the baseline rule (0.746). Against the rule the difference is **+0.008 ± 0.005** F1, ahead on 6 of the 7 splits, but inside the noise.
- At the default cut-off of 0.5 it would score 0.742, behind the rule. Choosing the cut-off on the training rows is worth **+0.012 ± 0.004** F1, ahead on 6 of the 7 splits.
- When many customers come back, predicting a repeat purchase for everyone already scores a high F1: in the block starting in March 2020 (repeat rate 0.60), always 1 (0.753) scores above the model at 0.5 (0.710).

## 4.4 Reading the Errors

The confusion matrix gathers the predictions of the 7 validation blocks. The table beside it shows the mean F1 of each cut-off on the training rows, where the cut-off is chosen, and on the validation rows, where it is checked.

In [6]:
both = pd.concat(pooled, ignore_index=True)
tn, fp, fn, tp = confusion_matrix(both["actual"], both["predicted"]).ravel()
print(f"{len(both):,} validation purchases: {tp} repeat purchases caught, {fn} missed,"
      f" {fp} false alarms, {tn} correctly left out\n")

curve = pd.concat(curves).groupby("cut-off").mean()
print(curve.loc[[0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]].round(3).to_string())
print(f"\nbest mean F1: at {curve['train F1'].idxmax()} on the training rows,"
      f" at {curve['validation F1'].idxmax()} on the validation rows")

4,116 validation purchases: 1857 repeat purchases caught, 457 missed, 744 false alarms, 1058 correctly left out

         train F1  validation F1
cut-off                         
0.30        0.748          0.744
0.35        0.755          0.751
0.40        0.762          0.753
0.45        0.760          0.753
0.50        0.749          0.742
0.55        0.729          0.733
0.60        0.708          0.705

best mean F1: at 0.41 on the training rows, at 0.42 on the validation rows


**Conclusions:**
- Of the 2,314 repeat purchases in the validation blocks, the model catches 1,857 (recall 0.80) and misses 457. Of its 2,601 positive predictions, 744 are false alarms (precision 0.71).
- The best mean cut-off on the training rows (0.41) is almost the best on the validation rows (0.42), and every value between 0.35 and 0.45 scores about the same on the validation rows (0.751 to 0.753): the choice carries over to new rows and does not hinge on a precise value. Above 0.5, F1 drops quickly.

## 4.5 Tuning the Logistic Regression

So far the logistic regression has scikit-learn's defaults: an L2 penalty with C=1, no class weights and an intercept. As in the course practical (week 6), a grid search tries every combination of these settings on the same 7 temporal splits:

- **the shape of the penalty:** L2, L1 or elastic net (`l1_ratio` 0, 1 or 0.5; the last two need the `saga` solver), or no penalty (`C=np.inf`);
- **`C`**, the inverse of the penalty's strength: 0.001, 0.01, 0.1, 1, 10 and 100;
- **`class_weight`:** none, or `"balanced"`;
- **`fit_intercept`:** with or without the intercept.

The recipe and the feature selection do not depend on these settings, so every combination is fitted on the matrices each split prepared in 4.2, and each one chooses its own cut-off on the training rows. AP decides, as in the other notebooks, and the F1 at each combination's cut-off is shown beside it.

In [7]:
# The grid of the course practical (week 6): three shapes of the penalty and six strengths, with
# and without class weights and the intercept, plus the model with no penalty (C=inf), which has
# no strength or shape left to tune
shapes = {"L2": {"l1_ratio": 0.0},
          "L1": {"l1_ratio": 1.0, "solver": "saga"},
          "elastic net": {"l1_ratio": 0.5, "solver": "saga"}}
grid = [{"penalty": name, "C": C, "class_weight": weight, "fit_intercept": intercept, **shape}
        for name, shape in shapes.items() for C in [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
        for weight in [None, "balanced"] for intercept in [True, False]]
grid += [{"penalty": "none", "C": np.inf, "class_weight": weight, "fit_intercept": intercept}
         for weight in [None, "balanced"] for intercept in [True, False]]

# The defaults used so far (make_model): L2, C=1, no class weights, with the intercept
default = next(index for index, config in enumerate(grid)
               if config["penalty"] == "L2" and config["C"] == 1.0
               and config["class_weight"] is None and config["fit_intercept"])


def make_tuned(config):
    """A fresh, unfitted logistic regression with one combination of the grid."""
    settings = {key: value for key, value in config.items() if key != "penalty"}
    return LogisticRegression(max_iter=5000, random_state=RANDOM_STATE, **settings)


def config_name(index):
    """One combination of the grid, as text."""
    config = grid[index]
    return (f"{config['penalty']}, C={config['C']:g}, class_weight={config['class_weight']}, "
            f"intercept={config['fit_intercept']}")


print(f"{len(grid)} combinations; the defaults are {config_name(default)}")

76 combinations; the defaults are L2, C=1, class_weight=None, intercept=True


In [8]:
# Every combination, fitted on the matrices each split prepared in 4.2
with warnings.catch_warnings():
    # saga can stop at max_iter with the weakest penalties, and C=inf prints scikit-learn's notice
    # that no penalty ignores C and l1_ratio; neither changes the comparison
    warnings.simplefilter("ignore", ConvergenceWarning)
    warnings.filterwarnings("ignore", message="Setting penalty=None")
    scores = score_configurations(prepared, grid, make_tuned)
mean_ap = scores.groupby("configuration")["AP"].mean()

# Mean AP of each shape and strength, without class weights and with the intercept
sweep = pd.DataFrame([{**config, "AP": mean_ap[index]} for index, config in enumerate(grid)])
display(sweep[sweep["class_weight"].isna() & sweep["fit_intercept"]]
        .pivot(index="C", columns="penalty", values="AP").round(4))

# The five best combinations and the defaults, each paired with the defaults on the same splits
reference_run = scores[scores["configuration"] == default].set_index("split")
board = {}
for index in dict.fromkeys(list(mean_ap.nlargest(5).index) + [default]):
    run = scores[scores["configuration"] == index].set_index("split")
    r = paired_delta(run["AP"], reference_run["AP"])
    board[config_name(index) + (" (defaults)" if index == default else "")] = {
        "AP": run["AP"].mean(), "dAP": r["delta"], "AP SEM": r["SEM"], "AP better on": r["wins"],
        "AP verdict": r["verdict"], "F1 at its cut-off": run["F1"].mean(),
        "dF1": run["F1"].mean() - reference_run["F1"].mean()}
pd.DataFrame.from_dict(board, orient="index").round(4)

penalty,L1,L2,elastic net,none
C,,,,
0.001,0.6283,0.8348,0.7857,NaN
0.010,0.8379,0.8403,0.8384,NaN
0.100,0.8431,0.8426,0.8429,NaN
1.000,0.8421,0.8421,0.8421,NaN
10.000,0.8421,0.8420,0.8421,NaN
100.000,0.8420,0.8420,0.8420,NaN
inf,NaN,NaN,NaN,0.842


,AP,dAP,AP SEM,AP better on,AP verdict,F1 at its cut-off,dF1
"L1, C=0.1, class_weight=balanced, intercept=True",0.8433,0.0012,0.0009,6/7,inside the noise,0.7523,-0.0019
"elastic net, C=0.1, class_weight=balanced, intercept=True",0.8432,0.0010,0.0005,6/7,inside the noise,0.7527,-0.0015
"L1, C=0.1, class_weight=None, intercept=True",0.8431,0.0009,0.0009,6/7,inside the noise,0.7515,-0.0027
"elastic net, C=0.1, class_weight=None, intercept=True",0.8429,0.0008,0.0005,6/7,inside the noise,0.7531,-0.0012
"L2, C=0.1, class_weight=balanced, intercept=True",0.8429,0.0007,0.0004,6/7,better,0.7518,-0.0025
"L2, C=1, class_weight=None, intercept=True (defaults)",0.8421,0.0000,0.0000,0/7,identical on every split,0.7543,0.0000


**Conclusions:**
- The sweep shows the pattern of the course practical: a heavy penalty (C=0.001) costs AP, most of all with L1 (0.628) and elastic net (0.786), which set weights to zero, while L2 only shrinks them (0.835). From C=0.1 on, the three shapes sit between 0.842 and 0.843, and with no penalty at all (C=inf) AP is 0.842: once the penalty is light, its strength and its shape barely matter.
- The best combination is L1 with C=0.1 and balanced class weights (AP 0.8433), only +0.0012 above the defaults, inside the noise. All five leaders have C=0.1 and keep the intercept; one of them (L2, C=0.1, balanced) is above the defaults by more than two standard errors, but by only +0.0007.
- None of them improves F1: at their own cut-offs they score 0.0012 to 0.0027 below the defaults (0.7543).

**Scoring the search on splits it did not see.** The best combination above is the best of 76 on the same 7 splits, so its score carries the luck of being chosen, like the k of RFE in the feature-selection notebook. The course estimates a search with nested cross-validation. With splits in time, the nested version walks forward: for each split from the third on, the combination with the best mean AP on the earlier splits is chosen and scored on that split, which took no part in the choice. That score is paired with the defaults' on the same splits, and the search is adopted only if it beats them by more than two standard errors; otherwise the simplest option, the defaults, stays.

In [9]:
# For each split from the third on: the combination chosen on the earlier splits, scored on it
honest = walk_forward(scores, first=2)
on_same = reference_run.loc[honest["split"]]
display(pd.DataFrame({
    "validation from": assessment.loc[honest["split"], "validation from"].to_numpy(),
    "chosen on the earlier splits": [config_name(index) for index in honest["chosen"]],
    "AP": honest["AP"].to_numpy(), "AP of the defaults": on_same["AP"].to_numpy(),
    "F1": honest["F1"].to_numpy(), "F1 of the defaults": on_same["F1"].to_numpy()}).round(4))

# The search minus the defaults on those splits
search_vs_defaults = pd.DataFrame({metric: paired_delta(honest[metric].to_numpy(), on_same[metric].to_numpy())
                                   for metric in ("AP", "F1")}).T
display(search_vs_defaults)

# The search is adopted only if it beats the defaults by more than two standard errors
chosen = int(mean_ap.idxmax()) if search_vs_defaults.loc["AP", "verdict"] == "better" else default
make_final_model = lambda: make_tuned(grid[chosen])
print(f"best combination on all 7 splits: {config_name(int(mean_ap.idxmax()))}")
print(f"settings of the final model: {config_name(chosen)}")

,validation from,chosen on the earlier splits,AP,AP of the defaults,F1,F1 of the defaults
0,2021-03-06,"L2, C=0.1, class_weight=None, intercept=False",0.8534,0.8539,0.7595,0.7685
1,2021-11-13,"L2, C=0.1, class_weight=balanced, intercept=True",0.8252,0.8249,0.7189,0.7213
2,2022-10-04,"elastic net, C=0.1, class_weight=balanced, int...",0.8195,0.8190,0.7336,0.7281
3,2023-07-29,"elastic net, C=0.1, class_weight=balanced, int...",0.8758,0.8743,0.7921,0.7909
4,2024-04-13,"L1, C=0.1, class_weight=balanced, intercept=True",0.8646,0.8626,0.7927,0.8005


,delta,SEM,wins,verdict
AP,0.000738,0.000425,4/5,inside the noise
F1,-0.00255,0.002719,2/5,inside the noise


best combination on all 7 splits: L1, C=0.1, class_weight=balanced, intercept=True
settings of the final model: L2, C=1, class_weight=None, intercept=True


**Conclusions:**
- Chosen on the earlier splits only, the search picks C=0.1 every time, with a different shape, weighting or intercept from one split to the next. On the 5 splits it did not see, it scores AP +0.0007 ± 0.0004 against the defaults (better on 4 of 5) and F1 −0.0026 ± 0.0027 (better on 2 of 5), both inside the noise.
- The search does not beat the defaults, so the final model keeps them (L2, C=1, no class weights, with the intercept). This is also the conclusion of the course practical, where the nested estimate of the search fell below the defaults.
- The settings of the logistic regression move AP by a few thousandths, far less than the differences between periods, so they are not where the score can improve.

## 4.6 Final Model and Kaggle Submission

The final model repeats the steps of each split once, on every training row. The feature-selection tables already hold them: `feature_selection_l1.csv` carries the training rows prepared by the recipe and reduced to the columns the feature-selection strategy kept, both fitted on every training row, and `feature_selection_l1_test.csv` carries the test rows prepared with those same fits. The model has the settings chosen in 4.5, the cut-off is chosen on the training rows, as in each split, and the submission keeps `ID` and writes 0/1 labels. Each new submission gets the next version number in its file name.

In [10]:
train_table = pd.read_csv("../../data/featureselection/feature_selection_l1.csv")
test_table = pd.read_csv("../../data/featureselection/feature_selection_l1_test.csv")
features = [c for c in train_table.columns if c not in ("ID", "customer_id", target)]
# The test table must carry the same columns, in the same order
assert list(test_table.columns) == list(train_table.columns.drop(target))

final_model, _, final_cutoff = fit_with_cutoff(make_final_model(), train_table[features], train_table[target])
test_probabilities = final_model.predict_proba(test_table[features])[:, 1]
submission = pd.DataFrame({"ID": test_table["ID"],
                           target: (test_probabilities >= final_cutoff).astype(int)})

# The same rows and IDs as sample_submission.csv, with labels rather than probabilities
assert list(submission["ID"]) == list(sample["ID"]) and set(submission[target]) <= {0, 1}

submission_file = Path("../../data/submissions/ML_Group_18_Version01.csv")
submission_file.parent.mkdir(exist_ok=True)
submission.to_csv(submission_file, index=False)
print(f"{len(features)} features | cut-off {final_cutoff} | {submission[target].mean():.1%} of the"
      f" test rows predicted to come back | saved to {submission_file.as_posix()}")

40 features | cut-off 0.4 | 76.7% of the test rows predicted to come back | saved to ../../data/submissions/ML_Group_18_Version01.csv


## 4.7 Conclusions

- **Model:** a logistic regression with scikit-learn's default settings, kept after the search of 4.5, on the 40 columns the feature-selection strategy keeps (the multicollinearity filter at 0.90, then L1), with the cut-off chosen on the training rows (0.40 on every training row).
- **Assessed performance:** F1 **0.754 ± 0.012** over the 7 temporal splits, from 0.721 to 0.801 depending on the period, against 0.746 for the baseline rule and 0.719 for always 1; the difference from the rule (+0.008) is inside the noise. This is the performance the Kaggle result should be read against.
- **Tuning:** a grid of 76 combinations (the shape and strength of the penalty, class weights and intercept), scored on splits that took no part in the choice, does not beat the defaults (AP +0.0007, F1 −0.0026, both inside the noise).
- **Submission:** `ML_Group_18_Version01.csv` holds the 986 test IDs; 76.7% of them are predicted to come back.
- **Limits:** the recipe, the feature-selection strategy and the cut-off rule were all decided on these same 7 splits, so the estimate is slightly optimistic. Only one model is assessed here; the final project compares at least five, with their hyperparameters tuned inside the temporal splits.